# Model fitting and city-wide grid prediction

Land use regression (LUR) experiment: varying LCS fraction with MLR, GAM, and RF models.

**Feature selection** uses Spearman-ordered sequential forward selection with a variance inflation admission test: features are ranked by |Spearman correlation with $y$| and added greedily; a candidate is admitted only if its variance inflation factor against the already selected features stays at or below `vif_threshold`. A backward pass then prunes the features that only became redundant once later candidates were admitted.

In [ ]:
import warnings

import dask
import numpy as np
import pandas as pd
import shap
import xarray as xr
from dask.diagnostics import ProgressBar
from flaml.default import RandomForestRegressor
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from zurich_uhi_lur import regr_utils
from zurich_uhi_lur.regr_utils import SpearmanVIFSelector

In [ ]:
# input data paths
stations_gdf_filepath = "../data/processed/stations.gpkg"
station_features_filepath = "../data/processed/stations-features.csv"
ts_df_filepath = "../data/processed/ts-df.csv"
grid_gdf_filepath = "../data/processed/regular-grid.gpkg"
grid_features_filepath = "../data/processed/grid-features.csv"

# output
dst_results_filepath = "../data/processed/lur-results.csv"
dst_importance_filepath = "../data/processed/lur-importance.csv"
dst_obs_pred_filepath = "../data/processed/lur-obs-pred.csv"
dst_shap_filepath = "../data/processed/lur-shap.csv"
dst_grid_pred_filepath = "../data/processed/lur-grid-pred.nc"
dst_grid_pred_pct_filepath = "../data/processed/lur-grid-pred-pct.csv"

# experiment settings
# hour of the day (local time)
target_hour = 23
n_reps = 5
seed = 0
fracs = [0, 0.33, 0.66, 1]
n_splits = 3
# the buffer scales of a given predictor are nested and hence strongly correlated (e.g.,
# building volume at 250 and 500 m correlate at 0.93), so a permissive collinearity
# criterion lets several scales of the same predictor coexist and take compensating
# signs. A variance inflation factor of 5 is the usual bar in the ecological
# collinearity literature (Dormann et al. 2013)
vif_threshold = 5.0
rsa_permutations = 999

# dask parallelism
# ACHTUNG: each worker holds its own copy of the pooled design matrix, so keep this low
# unless there is real memory headroom
n_dask_workers = 1

## Data loading


In [ ]:
station_features_gdf = regr_utils.get_site_features_gdf(
    station_features_filepath, stations_gdf_filepath
)
# Station 534 is co-located with Zürich-Kaserne (identical features, distorts KNN
# spatial weights for Moran I), so it is excluded from training and all diagnostics.
station_features_gdf = station_features_gdf[station_features_gdf.index != "534"]
aws_gdf = station_features_gdf[station_features_gdf["source"] == "AWS"]
lcs_gdf = station_features_gdf[station_features_gdf["source"] == "LCS"]

# Prediction grid — used to render the per-fraction prediction maps
grid_features_gdf = regr_utils.get_site_features_gdf(
    grid_features_filepath, grid_gdf_filepath
)
station_features_gdf.head()

In [ ]:
ts_df = (
    pd.read_csv(ts_df_filepath, index_col=["heatwave", "time"], parse_dates=["time"])
    .rename_axis(columns=station_features_gdf.index.name)
    .droplevel("heatwave")
)
# timestamps are in UTC, select the target hour in local time
ts_df = ts_df.loc[
    ts_df.index.get_level_values("time")
    .tz_localize("UTC")
    .tz_convert("Europe/Zurich")
    .hour
    == target_hour
]
ts_df.head()

## Model pipelines

Pipeline: `StandardScaler → SpearmanVIFSelector → Model`

Features are scaled first so that the Spearman correlation and the variance inflation
factors are computed on a comparable scale across features.

In [ ]:
def make_pipeline(model):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            ("fsel", SpearmanVIFSelector(vif_threshold=vif_threshold)),
            ("model", model),
        ]
    )


def _mlr_importances(pipe, X, selected):
    return dict(zip(selected, pipe.named_steps["model"].coef_))


def _ebm_shap_vals(pipe, X):
    """EBM: AdditiveExplainer via shap.Explainer auto-detection."""
    X_transformed = pipe[:-1].transform(X)
    explainer = shap.explainers.Additive(
        pipe.named_steps["model"].predict, shap.maskers.Independent(X_transformed)
    )
    return X_transformed, explainer(X_transformed).values


def _rf_shap_vals(pipe, X):
    """RF: TreeExplainer SHAP values. Shape: (n_samples, n_selected_features)."""
    X_transformed = pipe[:-1].transform(X)
    return X_transformed, shap.TreeExplainer(pipe.named_steps["model"])(
        X_transformed
    ).values


# Alternative global scalar: sign(Spearman(x_j, SHAP_j)) × mean(|SHAP_j|).
# Preserves mean(|SHAP|) magnitude while recovering direction via Spearman sign.
# def _signed_shap_importances(pipe, X, selected, shap_fn):
#     X_transformed, shap_values = shap_fn(pipe, X)
#     return {
#         feat: float(
#             np.sign(spearmanr(X_transformed[:, j], shap_values[:, j]).statistic)
#             * np.abs(shap_values[:, j]).mean()
#         )
#         for j, feat in enumerate(selected)
#     }


# (pipeline factory, importance extractor, shap fn) per model
MODELS = {
    "MLR": (lambda: make_pipeline(LinearRegression()), _mlr_importances, None),
    "GAM": (
        lambda: make_pipeline(
            ExplainableBoostingRegressor(interactions=0, random_state=seed)
        ),
        None,
        _ebm_shap_vals,
    ),
    "RF": (lambda: make_pipeline(RandomForestRegressor()), None, _rf_shap_vals),
}

## Experiment: varying LCS fraction

In [ ]:
rng = np.random.default_rng(seed)
kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

station_samples = []
for frac in fracs:
    if np.isclose(frac, 1):
        gdf = pd.concat([aws_gdf, lcs_gdf]).drop(columns="source")
        station_samples.append((frac, 0, gdf))
    else:
        for rep in range(n_reps):
            gdf = pd.concat(
                [aws_gdf, lcs_gdf.sample(frac=frac, random_state=rng)]
            ).drop(columns="source")
            station_samples.append((frac, rep, gdf))


def _run_trial(
    frac, rep, station_gdf, model_name, get_pipeline, importance_fn, shap_fn, seed
):
    """Single (frac, rep, model) trial."""
    station_gser = station_gdf["geometry"]
    y_with_time = (
        ts_df[station_gdf.index]
        .apply(lambda row: row - row.mean(), axis="columns")
        .stack()
    )
    y_ser = y_with_time.droplevel("time")
    X_df = station_gdf.loc[y_ser.index].drop(columns="geometry")
    groups = y_ser.index

    scored = regr_utils.score_pipeline(
        get_pipeline(),
        X_df,
        y_ser,
        cv=kf,
        station_gser=station_gser,
        groups=groups,
        rsa_permutations=rsa_permutations,
        importance_fn=importance_fn,
        rng=seed,
    )
    result = {
        "frac": frac,
        "rep": rep,
        "model": model_name,
        **scored,
        "y_obs": y_ser.values,
        "station_ids": np.asarray(groups),
    }
    if shap_fn is not None:
        # Compute SHAP on unique station features (not repeated timesteps).
        # Shape: (n_stations, n_selected_features)
        X_unique = station_gdf.drop(columns="geometry")
        _, shap_values = shap_fn(scored["pipe"], X_unique)
        result["shap_values"] = shap_values
        result["shap_station_ids"] = station_gdf.index.tolist()
        result["shap_feature_values"] = X_unique[scored["selected_features"]].values
    return result


tasks = [
    dask.delayed(_run_trial)(frac, rep, gdf, name, factory, imp_fn, shap_fn, seed)
    for frac, rep, gdf in station_samples
    for name, (factory, imp_fn, shap_fn) in MODELS.items()
]

# the k-nearest-neighbor weights of some station samples have disconnected
# components, which libpysal warns about for each Moran's I computation
warnings.filterwarnings("ignore", message="The weights matrix is not fully connected")
with ProgressBar():
    results = list(dask.compute(*tasks, num_workers=n_dask_workers))

# Expand frac=1 results to n_reps
results = [
    {**r, "rep": rep}
    for r in results
    for rep in (range(n_reps) if np.isclose(r["frac"], 1) else [r["rep"]])
]
_SCALAR = ["frac", "rep", "model", "r2", "r2_cv", "moran_I", "moran_p_sim"]
results_df = pd.DataFrame([{k: r.get(k) for k in _SCALAR} for r in results])

importance_df = pd.DataFrame(
    [
        {
            "frac": r["frac"],
            "rep": r["rep"],
            "model": r["model"],
            "feature": f,
            "value": v,
        }
        for r in results
        for f, v in r.get("importances", {}).items()
    ]
)

# Build obs-pred DataFrame — one row per (station_id × timestep) per trial
source_map = pd.concat(
    [
        aws_gdf[[]].assign(source="AWS"),
        lcs_gdf[[]].assign(source="LCS"),
    ]
)["source"]

obs_pred_df = pd.DataFrame(
    [
        {
            "frac": r["frac"],
            "rep": r["rep"],
            "model": r["model"],
            "station_id": sid,
            "obs": float(obs_val),
            "pred": float(pred_val),
            "source": source_map.get(sid, "LCS"),
        }
        for r in results
        for sid, obs_val, pred_val in zip(r["station_ids"], r["y_obs"], r["y_pred_cv"])
    ]
)


shap_df = pd.DataFrame(
    [
        {
            "frac": r["frac"],
            "rep": r["rep"],
            "model": r["model"],
            "station_id": sid,
            "feature": feat,
            "shap_value": float(r["shap_values"][k, j]),
            "feature_value": float(r["shap_feature_values"][k, j]),
        }
        for r in results
        if "shap_values" in r
        for k, sid in enumerate(r["shap_station_ids"])
        for j, feat in enumerate(r["selected_features"])
    ]
)

frac_pipelines = {
    frac: {
        r["model"]: r["pipe"]
        for r in results
        if np.isclose(r["frac"], frac) and r["rep"] == 0
    }
    for frac in fracs
}
final_pipelines = frac_pipelines[fracs[-1]]
final_features = {
    r["model"]: r["selected_features"]
    for r in results
    if np.isclose(r["frac"], fracs[-1]) and r["rep"] == 0
}

In [ ]:
# Grid predictions per (frac, rep, model), aggregated over reps.
# Persisting the mean/std raster (a few MB) avoids saving every fitted pipeline
# just to re-predict the grid downstream. The spread across reps captures the
# sampling uncertainty from drawing different LCS subsets at each fraction.
X_grid = grid_features_gdf.drop(columns=["geometry", "source"], errors="ignore")
grid_y = grid_features_gdf.geometry.y.values
grid_x = grid_features_gdf.geometry.x.values

_grid_pred_cache = {}


def _grid_pred_da(pipe):
    # cache by identity: frac=1 reps reuse the same fitted pipeline object
    key = id(pipe)
    if key not in _grid_pred_cache:
        preds = pipe.predict(X_grid)
        _grid_pred_cache[key] = (
            pd.DataFrame({"y": grid_y, "x": grid_x, "pred": preds})
            .set_index(["y", "x"])["pred"]
            .to_xarray()
        )
    return _grid_pred_cache[key]


model_order = list(MODELS)
mean_by_model, std_by_model = [], []
# ACHTUNG: the spread of the predicted temperature range across repetitions cannot be
# recovered downstream from the rep-mean raster, since averaging the surfaces shrinks
# their extremes, so the percentiles are taken here, per rep, before aggregating
PCTS = (5, 95)
pct_rows = []
for model_name in model_order:
    mean_by_frac, std_by_frac = [], []
    for frac in fracs:
        reps_da = xr.concat(
            [
                _grid_pred_da(r["pipe"])
                for r in results
                if r["model"] == model_name and np.isclose(r["frac"], frac)
            ],
            dim="rep",
        )
        mean_by_frac.append(reps_da.mean("rep"))
        std_by_frac.append(reps_da.std("rep"))
        for rep_idx in range(reps_da.sizes["rep"]):
            rep_arr = reps_da.isel(rep=rep_idx).values
            p_lo, p_hi = np.percentile(rep_arr[np.isfinite(rep_arr)], PCTS)
            pct_rows.append(
                {
                    "model": model_name,
                    "frac": float(frac),
                    "rep": rep_idx,
                    "p5": p_lo,
                    "p95": p_hi,
                }
            )
    frac_idx = pd.Index(fracs, name="frac")
    mean_by_model.append(xr.concat(mean_by_frac, frac_idx))
    std_by_model.append(xr.concat(std_by_frac, frac_idx))

model_idx = pd.Index(model_order, name="model")
grid_pred_ds = xr.Dataset(
    {
        "pred_mean": xr.concat(mean_by_model, model_idx),
        "pred_std": xr.concat(std_by_model, model_idx),
    }
)
grid_pred_ds.attrs["crs"] = grid_features_gdf.crs.to_string()
grid_pred_ds

## Save results


In [ ]:
results_df.to_csv(dst_results_filepath, index=False)
importance_df.to_csv(dst_importance_filepath, index=False)
shap_df.to_csv(dst_shap_filepath, index=False)
obs_pred_df.to_csv(dst_obs_pred_filepath, index=False)

# Aggregated grid predictions (mean/std over reps) for the prediction maps.
# scipy engine writes NetCDF3, which is enough here and needs no extra deps.
grid_pred_ds.to_netcdf(dst_grid_pred_filepath, engine="scipy")
pd.DataFrame(pct_rows).to_csv(dst_grid_pred_pct_filepath, index=False)